# Select patients from stroke registry data
- Inclusion criteria: > 17y, ischemic stroke, inpatient/non-transferred, not refusing to participate
- Exclusion criteria: < 12h

In [238]:
import pandas as pd
import numpy as np
import os

In [239]:
stroke_registry_data_path = '/Users/jk1/OneDrive - unige.ch/stroke_research/geneva_stroke_unit_dataset/data/stroke_registry/post_hoc_modified/stroke_registry_post_hoc_modified.xlsx'

In [240]:
manual_eds_completion_folder = '/Users/jk1/OneDrive - unige.ch/stroke_research/geneva_stroke_unit_dataset/data/stroke_registry/manuel_eds_completion'

In [241]:
output_path = '/Users/jk1/temp/opsum_extration_output'

In [242]:
all_data_df = pd.read_excel(stroke_registry_data_path)

In [243]:
from preprocessing.utils import create_registry_case_identification_column

all_data_df['patient_id'] = all_data_df['Case ID'].apply(lambda x: x[8:-4]).astype(str)
all_data_df['EDS_last_4_digits'] = all_data_df['Case ID'].apply(lambda x: x[-4:]).astype(str)
all_data_df['case_admission_id'] = create_registry_case_identification_column(all_data_df)

In [244]:
n_duplicates = len(all_data_df[all_data_df['Type of event'] == 'duplicate']['case_admission_id'].unique())
n_records_screened = len(all_data_df['case_admission_id'].unique()) - n_duplicates

all_data_df = all_data_df[all_data_df['Type of event'] != 'duplicate']

print('Number of records screened: ', n_records_screened, 'after removing duplicates: ', n_duplicates)

Number of records screened:  5155 after removing duplicates:  9


### Exclude patients refusing participation in research

In [245]:
# Remove patients not wanting to participate in research
print(all_data_df['Patient refuses use of data for research'].value_counts())
n_patient_refuses_research = len(all_data_df[all_data_df['Patient refuses use of data for research'] == 'yes'])
full_data_df = all_data_df[all_data_df['Patient refuses use of data for research'] != 'yes']

yes    18
Name: Patient refuses use of data for research, dtype: int64


### Include only ischemic stroke patients

In [246]:
full_data_df['Type of event'].value_counts()

Ischemic stroke                          3415
Transient ischemic attack                 712
Stroke or TIA mimic                       498
Intracerebral hemorrhage                  381
Retinal infarct                            54
Amaurosis fugax                            47
Cerebral sinus vein thrombosis             27
Acute ischemic myelopathy                   4
Non-traumatic subarachnoid hemorrhage       1
Name: Type of event, dtype: int64

In [247]:
# all_data_df['Type of event'].value_counts().to_excel(os.path.join(output_path, 'type_of_event.xlsx'))

In [248]:
# select only ischemic stroke patients
all_stroke_df = full_data_df[full_data_df['Type of event'] == 'Ischemic stroke']

In [249]:
n_patients_not_ischemic_stroke = len(full_data_df['case_admission_id'].unique()) - len(all_stroke_df['case_admission_id'].unique())
print('Number of patients excluded because not ischemic stroke: ', n_patients_not_ischemic_stroke)

Number of patients excluded because not ischemic stroke:  1724


### Exclude patients not hospitalised in our center or discharged

In [250]:
all_stroke_df['Initial hospitalization'].value_counts()

Monitored stroke unit in-house                        2523
Other intermediate or intensive care unit in-house     548
Non-monitored stroke unit in-house                     212
Referral to other hospital or care institution          58
General ward in-house                                   34
Outpatient management                                   33
Referral to other Stroke Unit or Stroke Center           7
Name: Initial hospitalization, dtype: int64

In [251]:
# all_data_df['Initial hospitalization'].value_counts().to_excel(os.path.join(output_path, 'initial_hospitalization.xlsx'))

In [252]:
# exclude patients that were immediately discharged or referred to other center
stroke_df = all_stroke_df[all_stroke_df['Initial hospitalization'] != 'Outpatient management']
stroke_df = stroke_df[stroke_df['Initial hospitalization'] != 'Referral to other Stroke Unit or Stroke Center']
stroke_df = stroke_df[stroke_df['Initial hospitalization'] != 'Referral to other hospital or care institution']

In [253]:
n_not_hospitalised_in_house = len(all_stroke_df['case_admission_id'].unique()) - len(stroke_df['case_admission_id'].unique())
print('Number of patients excluded because discharged or referred to other center: ', n_not_hospitalised_in_house)

Number of patients excluded because discharged or referred to other center:  98


### Exclude patients with less than 12h of hospitalization

In [254]:
# set end of reference period to stroke onset or arrival at hospital, whichever is later
# this takes into account potential in-hospital stroke events

datatime_format = '%d.%m.%Y %H:%M'
stroke_df['arrival_dt'] = pd.to_datetime(stroke_df['Arrival at hospital'],
                                                  format='%Y%m%d').dt.strftime('%d.%m.%Y') + ' ' + \
                                   pd.to_datetime(stroke_df['Arrival time'], format='%H:%M',
                                                  infer_datetime_format=True).dt.strftime('%H:%M')

stroke_df['stroke_dt'] = pd.to_datetime(stroke_df['Onset date'],
                                                 format='%Y%m%d').dt.strftime('%d.%m.%Y') + ' ' + \
                                    pd.to_datetime(stroke_df['Onset time'], format='%H:%M',
                                                   infer_datetime_format=True).dt.strftime('%H:%M')

stroke_df['delta_onset_arrival'] = (
        pd.to_datetime(stroke_df['stroke_dt'], format=datatime_format, errors='coerce')
        - pd.to_datetime(stroke_df['arrival_dt'], format=datatime_format, errors='coerce')
).dt.total_seconds()
stroke_df['registry_sampling_start_upper_bound_reference'] = stroke_df \
    .apply(lambda x: x['stroke_dt'] if x['delta_onset_arrival'] > 0 else x['arrival_dt'],
           axis=1)


In [255]:
stroke_df['discharge_dt'] = pd.to_datetime(stroke_df['Discharge date'],
                                                  format='%Y%m%d').dt.strftime('%d.%m.%Y') + ' ' + \
                                   pd.to_datetime(stroke_df['Discharge time'], format='%H:%M',
                                                  infer_datetime_format=True).dt.strftime('%H:%M')

stroke_df['death_dt'] = pd.to_datetime(stroke_df['Death at hospital date'],
                                                  format='%Y%m%d').dt.strftime('%d.%m.%Y') + ' ' + \
                                   pd.to_datetime(stroke_df['Death at hospital time'], format='%H:%M',
                                                  infer_datetime_format=True).dt.strftime('%H:%M')

stroke_df['registry_sampling_end'] = stroke_df['discharge_dt'].fillna(stroke_df['death_dt'])


In [256]:
stroke_df['registry_sample_range'] = pd.to_datetime(stroke_df['registry_sampling_end'], format=datatime_format) \
                                                - pd.to_datetime(stroke_df['registry_sampling_start_upper_bound_reference'], format=datatime_format)

In [257]:
cid_with_hospitalization_duration_less_than_12h = stroke_df[stroke_df['registry_sample_range'] < pd.Timedelta('12h')]['case_admission_id'].unique()

In [258]:
n_with_hospitalization_duration_less_than_12h = len(cid_with_hospitalization_duration_less_than_12h)
print('Number of patients excluded because hospitalization duration less than 12h: ', len(cid_with_hospitalization_duration_less_than_12h))
print('NB: more patients will be excluded programmatically if total span of data is less than 12h')

Number of patients excluded because hospitalization duration less than 12h:  8
NB: more patients will be excluded programmatically if total span of data is less than 12h


In [259]:
# exclude patients with less than 12h of hospitalization
stroke_df = stroke_df[~stroke_df['case_admission_id'].isin(cid_with_hospitalization_duration_less_than_12h)]

In [260]:
exclude_transfers_from_france = False
if exclude_transfers_from_france:
    # find cids from transfers from France (where Non-Swiss == yes & referral == other hospital)
    cids_transfers_from_france = stroke_df[(stroke_df['Referral'] == 'Other hospital') & (stroke_df['Non-Swiss'] == 'yes')]['case_admission_id'].values
    stroke_df = stroke_df[~stroke_df['case_admission_id'].isin(cids_transfers_from_france)]

In [261]:
len(stroke_df['case_admission_id'].unique())

3307

In [262]:
# counting patients with outcome variables
sum(stroke_df['3M mRS'].value_counts())

2730

In [263]:
stroke_df['Death in hospital'].value_counts()


no     3128
yes     179
Name: Death in hospital, dtype: int64

In [264]:
stroke_df['Referral'].value_counts()

Emergency service (144)               1591
Other hospital                         721
Self referral                          512
General Practitioner                   249
In-hospital event                      210
Other Stroke Unit or Stroke Center      26
Name: Referral, dtype: int64

In [265]:
stroke_df['Initial hospitalization'].value_counts()

Monitored stroke unit in-house                        2518
Other intermediate or intensive care unit in-house     547
Non-monitored stroke unit in-house                     210
General ward in-house                                   34
Name: Initial hospitalization, dtype: int64

In [266]:
onset_date = pd.to_datetime(pd.to_datetime(stroke_df['Onset date'], format='%Y%m%d').dt.strftime('%d-%m-%Y') \
                                        + ' ' + stroke_df['Onset time'])

admission_date = pd.to_datetime(pd.to_datetime(stroke_df['Arrival at hospital'], format='%Y%m%d').dt.strftime('%d-%m-%Y') \
                                        + ' ' + stroke_df['Arrival time'])

discharge_date = pd.to_datetime(pd.to_datetime(stroke_df['Discharge date'], format='%Y%m%d').dt.strftime('%d-%m-%Y') \
                                        + ' ' + stroke_df['Discharge time'])


In [267]:
stroke_df.head()

,Unnamed: 0,Case ID,Centre,Entry date,Entry person,Patient refuses use of data for research,Last name,First name,DOB,Type of event,...,EDS_last_4_digits,case_admission_id,arrival_dt,stroke_dt,delta_onset_arrival,registry_sampling_start_upper_bound_reference,discharge_dt,death_dt,registry_sampling_end,registry_sample_range
3,4,SSR-HUG-1005030884,HUG Genève,20181108,Emmanuel Carrera,NaN,NaN,NaN,19331031.0,Ischemic stroke,...,0884,100503_0884,08.11.2018 13:21,08.11.2018 08:30,-17460.0,08.11.2018 13:21,16.11.2018 13:00,NaN,16.11.2018 13:00,7 days 23:39:00
5,6,SSR-HUG-10055644109,HUG Genève,20181002,Emmanuel Carrera,NaN,NaN,NaN,19570311.0,Ischemic stroke,...,4109,1005564_4109,02.10.2018 08:11,28.09.2018 00:00,-375060.0,02.10.2018 08:11,22.10.2018 09:35,NaN,22.10.2018 09:35,20 days 01:24:00
6,7,SSR-HUG-10057989217,HUG Genève,20180910,Emmanuel Carrera,NaN,NaN,NaN,19500315.0,Ischemic stroke,...,9217,1005798_9217,10.09.2018 21:36,08.09.2018 09:00,-218160.0,10.09.2018 21:36,18.09.2018 17:47,NaN,18.09.2018 17:47,7 days 20:11:00
11,12,SSR-HUG-10117940030,HUG Genève,20180122,Roman Sztajzel,NaN,NaN,NaN,19210723.0,Ischemic stroke,...,0030,1011794_0030,22.01.2018 13:13,22.01.2018 11:30,-6180.0,22.01.2018 13:13,NaN,01.02.2018 01:00,01.02.2018 01:00,9 days 11:47:00
13,14,SSR-HUG-10129157747,HUG Genève,20180520,Emmanuel Carrera,NaN,NaN,NaN,19350829.0,Ischemic stroke,...,7747,1012915_7747,20.05.2018 19:07,20.05.2018 16:00,-11220.0,20.05.2018 19:07,24.05.2018 12:37,NaN,24.05.2018 12:37,3 days 17:30:00


#### Fuse with databases of manually completed EDS

In [268]:
manual_eds_completion_dfs = [pd.read_excel(os.path.join(manual_eds_completion_folder, f)) for f in os.listdir(manual_eds_completion_folder) if f.endswith('.xlsx')]

In [269]:
all_manual_eds_completions = pd.concat(manual_eds_completion_dfs)

In [270]:
all_manual_eds_completions = all_manual_eds_completions[['patient_id', 'EDS_last_4_digits', 'manual_eds', 'manual_patient_id']]
all_manual_eds_completions = all_manual_eds_completions.astype(str)
all_manual_eds_completions['manual_patient_id'] = all_manual_eds_completions['manual_patient_id'].str.rstrip('.0')
all_manual_eds_completions['manual_eds'] = all_manual_eds_completions['manual_eds'].str.rstrip('.0')

In [271]:
all_manual_eds_completions.head()

,patient_id,EDS_last_4_digits,manual_eds,manual_patient_id
0,109264,5791,15151588,109264
1,2001,5697,14929722,200156
2,20463,2067,15081282,20463
3,758200,5847,15037633,7582
4,152696,4056,1559085,152696


In [272]:
stroke_df = stroke_df.merge(all_manual_eds_completions, how='left', on=['patient_id', 'EDS_last_4_digits'])

In [273]:
stroke_df.head()

,Unnamed: 0,Case ID,Centre,Entry date,Entry person,Patient refuses use of data for research,Last name,First name,DOB,Type of event,...,arrival_dt,stroke_dt,delta_onset_arrival,registry_sampling_start_upper_bound_reference,discharge_dt,death_dt,registry_sampling_end,registry_sample_range,manual_eds,manual_patient_id
0,4,SSR-HUG-1005030884,HUG Genève,20181108,Emmanuel Carrera,NaN,NaN,NaN,19331031.0,Ischemic stroke,...,08.11.2018 13:21,08.11.2018 08:30,-17460.0,08.11.2018 13:21,16.11.2018 13:00,NaN,16.11.2018 13:00,7 days 23:39:00,NaN,NaN
1,6,SSR-HUG-10055644109,HUG Genève,20181002,Emmanuel Carrera,NaN,NaN,NaN,19570311.0,Ischemic stroke,...,02.10.2018 08:11,28.09.2018 00:00,-375060.0,02.10.2018 08:11,22.10.2018 09:35,NaN,22.10.2018 09:35,20 days 01:24:00,NaN,NaN
2,7,SSR-HUG-10057989217,HUG Genève,20180910,Emmanuel Carrera,NaN,NaN,NaN,19500315.0,Ischemic stroke,...,10.09.2018 21:36,08.09.2018 09:00,-218160.0,10.09.2018 21:36,18.09.2018 17:47,NaN,18.09.2018 17:47,7 days 20:11:00,NaN,NaN
3,12,SSR-HUG-10117940030,HUG Genève,20180122,Roman Sztajzel,NaN,NaN,NaN,19210723.0,Ischemic stroke,...,22.01.2018 13:13,22.01.2018 11:30,-6180.0,22.01.2018 13:13,NaN,01.02.2018 01:00,01.02.2018 01:00,9 days 11:47:00,NaN,NaN
4,14,SSR-HUG-10129157747,HUG Genève,20180520,Emmanuel Carrera,NaN,NaN,NaN,19350829.0,Ischemic stroke,...,20.05.2018 19:07,20.05.2018 16:00,-11220.0,20.05.2018 19:07,24.05.2018 12:37,NaN,24.05.2018 12:37,3 days 17:30:00,NaN,NaN


In [274]:
selected_columns = ['patient_id', 'EDS_last_4_digits', 'manual_eds', 'manual_patient_id', 'DOB',
                                                   'Arrival at hospital', 'Arrival time',
                                                   'Discharge date', 'Discharge time',
                                                   'Death at hospital date', 'Death at hospital time', 'Time of symptom onset known', 'Onset date', 'Onset time', 'Referral']

In [275]:
extraction_target_df = stroke_df.copy()
# for extraction replace missing stroke onset date with admission_date (to have a reference date in case of in-hospital strokes)
extraction_target_df['Onset time'] = extraction_target_df.apply(lambda x: x['Arrival time'] if pd.isnull(x['Onset date']) else x['Onset time'], axis=1)
extraction_target_df['Onset date'] = extraction_target_df.apply(lambda x: x['Arrival at hospital'] if pd.isnull(x['Onset date']) else x['Onset date'], axis=1)

In [276]:
high_frequency_data_patient_selection_with_details = stroke_df[selected_columns]
extraction_target_df = extraction_target_df[selected_columns]

In [277]:
high_frequency_data_patient_selection_with_details.rename(columns={'Onset date': 'Stroke onset date', 'Onset time': 'Stroke onset time'}, inplace=True)
extraction_target_df.rename(columns={'Onset date': 'Stroke onset date', 'Onset time': 'Stroke onset time'}, inplace=True)


/Users/jk1/opt/anaconda3/envs/opsum/lib/python3.8/site-packages/pandas/core/frame.py:5039: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  return super().rename(


In [278]:
high_frequency_data_patient_selection_with_details.head()

,patient_id,EDS_last_4_digits,manual_eds,manual_patient_id,DOB,Arrival at hospital,Arrival time,Discharge date,Discharge time,Death at hospital date,Death at hospital time,Time of symptom onset known,Stroke onset date,Stroke onset time,Referral
0,100503,0884,NaN,NaN,19331031.0,20181108,13:21,20181116.0,13:00,NaN,NaN,no,20181108.0,08:30,Emergency service (144)
1,1005564,4109,NaN,NaN,19570311.0,20181002,08:11,20181022.0,09:35,NaN,NaN,no,20180928.0,00:00,Emergency service (144)
2,1005798,9217,NaN,NaN,19500315.0,20180910,21:36,20180918.0,17:47,NaN,NaN,yes,20180908.0,09:00,Other hospital
3,1011794,0030,NaN,NaN,19210723.0,20180122,13:13,NaN,NaN,20180201.0,01:00,yes,20180122.0,11:30,Emergency service (144)
4,1012915,7747,NaN,NaN,19350829.0,20180520,19:07,20180524.0,12:37,NaN,NaN,yes,20180520.0,16:00,Emergency service (144)


In [279]:
import pandas as pd
# excluded patients logs
excluded_patients_df = pd.DataFrame({
    'n_records_screened': n_records_screened,
    'n_patient_refuses_research': n_patient_refuses_research,
    'n_patients_not_ischemic_stroke': n_patients_not_ischemic_stroke,
    'n_not_hospitalised_in_house': n_not_hospitalised_in_house,
    'n_with_hospitalization_duration_less_than_12h': n_with_hospitalization_duration_less_than_12h,
    'Comments': 'more patients will be excluded programmatically (1. insufficient length of hosp, 2. patient not found in EHR)'
}, index=[0])

excluded_patients_df = excluded_patients_df.T
excluded_patients_df.columns = ['number of patients']

In [280]:
save_data = True

In [281]:
import time

if save_data:
    high_frequency_data_patient_selection_with_details.to_csv(os.path.join(output_path, 'high_frequency_data_patient_selection_with_details.csv'))
    extraction_target_df.to_csv(os.path.join(output_path, 'high_frequency_data_patient_selection_extraction_target.csv'))

    timestamp = time.strftime("%d%m%Y_%H%M%S")
    excluded_patients_df.to_csv(os.path.join(output_path, f'excluded_patients_df_{timestamp}.tsv'), sep='\t')